# PMCO 01 | 사진 특징 → 음악 계획 → (선택) AI 음악 생성
강원대학교 오픈소스프로그래밍 실습: **로컬 특성 분석은 무료**, Gemini/Lyria API는 **실제 요금이 발생할 수 있습니다**.
이 노트북은 독립 실행형이며, 생성된 코드를 GitHub에 저장할 수 있습니다.

## ① 준비 — Colab에서 셀 실행
왼쪽 ▶ 버튼 또는 Shift+Enter. 첫 셀을 실행해 라이브러리를 설치합니다.

In [ ]:
!pip -q install -U numpy pillow pandas matplotlib requests beautifulsoup4 google-genai

## ② 매핑 알고리즘 파일 만들기
`%%writefile pmco_core.py`는 아래 Python 코드를 실제 파일로 저장합니다. GitHub에 노트북을 저장해도 이 셀을 재실행하면 파일이 다시 만들어집니다.

In [ ]:
%%writefile pmco_core.py
"""PMCO: interpretable prototype image-feature to music-parameter mapping.

These heuristics are design hypotheses, NOT scientifically validated perceptual laws.
"""
from __future__ import annotations
from dataclasses import dataclass, asdict
from io import BytesIO
from typing import Optional
import numpy as np
from PIL import Image

@dataclass
class ImageFeatures:
    brightness: float
    saturation: float
    contrast: float
    complexity: float
    warmth: float

@dataclass
class MusicPlan:
    energy: float
    valence: float
    bpm: int
    genre: str
    instrument: str
    dynamics: str
    explanation: dict

def _clip01(n):
    return float(np.clip(n, 0.0, 1.0))

def extract_features(image: Image.Image | bytes) -> ImageFeatures:
    """Return five [0,1] proxies. 'complexity' is local edge magnitude, not scene motion."""
    if isinstance(image, bytes):
        image = Image.open(BytesIO(image))
    image = image.convert('RGB')
    image.thumbnail((512, 512))
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    sat = np.asarray(image.convert('HSV'), dtype=np.float32)[..., 1] / 255.0
    diff_h = np.abs(np.diff(y, axis=1)).mean() if y.shape[1] > 1 else 0.0
    diff_v = np.abs(np.diff(y, axis=0)).mean() if y.shape[0] > 1 else 0.0
    # Approximate scaling is a project hypothesis; adjust using labeled data.
    return ImageFeatures(
        brightness=_clip01(y.mean()),
        saturation=_clip01(sat.mean()),
        contrast=_clip01(y.std() / 0.35),
        complexity=_clip01((diff_h + diff_v) / 0.28),
        warmth=_clip01(0.5 + float((r - b).mean()) * 0.9),
    )

def map_music(features: ImageFeatures, energy_override: Optional[float]=None,
              weights: tuple[float,float,float]=(0.40, 0.35, 0.25)) -> MusicPlan:
    """Transparent deterministic mapping: complexity, saturation, contrast -> energy."""
    w = np.asarray(weights, dtype=float)
    if not np.isfinite(w).all() or (w < 0).any() or w.sum() <= 0:
        raise ValueError('weights must be nonnegative and sum to a positive value')
    w /= w.sum()
    contributions = {
        'complexity': float(w[0] * features.complexity),
        'saturation': float(w[1] * features.saturation),
        'contrast': float(w[2] * features.contrast),
    }
    raw = sum(contributions.values())
    energy = _clip01(raw if energy_override is None else energy_override)
    valence = _clip01(0.45 * features.brightness + 0.35 * features.warmth + 0.20 * features.saturation)
    bpm = round(55 + 95 * energy)
    if energy < 0.33:
        genre, dynamics = 'ambient / minimalist', 'soft, slow development'
        instrument = 'warm piano, airy pads' if features.warmth > 0.5 else 'felt piano, soft strings'
    elif energy < 0.67:
        genre, dynamics = 'acoustic / cinematic', 'gentle pulse, moderate intensity'
        instrument = 'acoustic guitar, strings' if features.warmth > 0.5 else 'piano, light percussion'
    else:
        genre, dynamics = 'cinematic / energetic', 'rhythmic, lively accents'
        instrument = 'bright percussion, plucked strings' if features.warmth > 0.5 else 'synths, drums'
    return MusicPlan(energy=energy, valence=valence, bpm=bpm, genre=genre,
                     instrument=instrument, dynamics=dynamics, explanation={
                         'weight_complexity': round(float(w[0]), 3),
                         'weight_saturation': round(float(w[1]), 3),
                         'weight_contrast': round(float(w[2]), 3),
                         'weighted_complexity': round(contributions['complexity'], 4),
                         'weighted_saturation': round(contributions['saturation'], 4),
                         'weighted_contrast': round(contributions['contrast'], 4),
                         'computed_energy_before_override': round(raw, 4),
                         'energy_overridden': energy_override is not None,
                     })

def build_music_prompt(plan: MusicPlan, scene_description: str='') -> str:
    scene = scene_description.strip().replace('\n', ' ')[:500]
    return (
        'Create an original instrumental music clip. No vocals, no lyrics, '
        'no imitation of copyrighted recordings or named artists. '
        f'Target tempo approximately {plan.bpm} BPM. '
        f'Energy {plan.energy:.2f}/1, emotional brightness {plan.valence:.2f}/1. '
        f'Style: {plan.genre}. Instruments: {plan.instrument}. '
        f'Dynamics: {plan.dynamics}. '
        'Express the visual atmosphere with musical texture, not literal sounds. '
        + (f'Image scene notes: {scene}.' if scene else '')
    )

def plan_record(filename: str, features: ImageFeatures, plan: MusicPlan) -> dict:
    base={'filename':filename, **asdict(features)}
    for k,v in asdict(plan).items():
        if k != 'explanation': base[k] = v
    for k,v in plan.explanation.items(): base[k] = v
    return base


## ③ 개인 사진 업로드
공개 가능한 사진만 사용하세요. 고요한 풍경, 활기찬 시장 등 분위기가 다른 사진 두 장을 비교하면 좋습니다.

In [ ]:
from google.colab import files
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from pmco_core import extract_features, map_music, build_music_prompt, plan_record
uploaded=files.upload()
if not uploaded: raise ValueError('사진 파일 한 장을 업로드해 줘.')
filename, image_bytes=next(iter(uploaded.items()))
photo=Image.open(BytesIO(image_bytes)).convert('RGB')
display(photo.resize((min(photo.width,700), int(photo.height*min(photo.width,700)/photo.width))))

## ④ 시각 특징 5종 계산
- 밝기/채도/대비/경계 복잡도/붉은색-푸른색 상대 비율(색온감의 간단한 대리지표)
- **움직임** 자체를 인식하지 않는다는 점에 주의!

In [ ]:
import pandas as pd
features=extract_features(photo)
display(pd.DataFrame([vars(features)]).T.rename(columns={0:'normalized_value'}))

## ⑤ 명시적 음악 매핑: E=0.40×복잡도+0.35×채도+0.25×대비, BPM=55+95E
가중치는 **우리가 검증할 초기 가설**입니다. 과학적 고정 법칙이 아닙니다.

In [ ]:
music=map_music(features)
print('BPM:',music.bpm,'| Energy:',round(music.energy,3))
print('Genre:',music.genre,'| Instruments:',music.instrument)
print('각 특징의 기여도:',music.explanation)
prompt=build_music_prompt(music)
print('\n음악 생성 프롬프트:\n',prompt)

## ⑥ 사용자 개입 실험: 같은 사진으로 에너지만 변경
출력이 바뀌는지 확인하고 나중에 A/B 비교 시 이용합니다.

In [ ]:
cases=[map_music(features,energy_override=x) for x in [0.20,0.50,0.85]]
display(pd.DataFrame([{'energy_override':x,'BPM':p.bpm,'genre':p.genre,'instrument':p.instrument} for x,p in zip([0.20,0.50,0.85],cases)]))

## ⑦ (선택) Gemini에게 사진 분위기 설명받기 — 유료 호출 가능
왼쪽 열쇠(Secrets)에 `GEMINI_API_KEY`를 저장하고 노트북 접근을 허용하세요. **키 문자열을 이 노트북에 직접 입력하지 마세요.**
처음에는 `USE_GEMINI = False`로 실행하세요.

In [ ]:
USE_GEMINI = False  # 사용자가 True로 변경해야 실제 유료 API 호출
scene_description = ''
if USE_GEMINI:
    from google.colab import userdata
    from google import genai
    from google.genai import types
    client=genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    from mimetypes import guess_type
    mime=guess_type(filename)[0] or 'image/jpeg'
    response=client.models.generate_content(
        model='gemini-3.8-flash',
        contents=[types.Part.from_bytes(data=image_bytes,mime_type=mime),
                  '사진에서 관찰 가능한 장면과 고요함/역동성, 계절, 분위기를 한국어로 2문장 설명해줘. 불확실한 내용은 추정이라고 표현해.'],
    )
    scene_description=response.text or ''
    print(scene_description)
else:
    print('Gemini 호출 생략: 비용 없이 수치 알고리즘을 테스트합니다.')
prompt=build_music_prompt(music,scene_description)
print('\n최종 프롬프트:\n',prompt)

## ⑧ (선택) Lyria 3.5로 실제 MP3 만들기 — 유료 호출
모델 `lyria-3.5-clip-preview`가 계정/지역에서 활성화되어 있는지 공식 문서와 가격표를 확인하세요.
**호출하려는 경우만** `GENERATE_PAID_MUSIC = True`로 바꾸세요.

In [ ]:
GENERATE_PAID_MUSIC = False  # 기본 비활성, 의도한 유료 실행만 True
if GENERATE_PAID_MUSIC:
    import base64
    from IPython.display import Audio, display
    from google.colab import userdata
    from google import genai
    client=genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    interaction=client.interactions.create(model='lyria-3.5-clip-preview',input=prompt)
    audio=interaction.output_audio
    if audio is None or not audio.data:
        raise RuntimeError('생성된 오디오가 없습니다. 모델 사용 권한/응답을 확인하세요.')
    data=base64.b64decode(audio.data) if isinstance(audio.data,str) else audio.data
    mime=(audio.mime_type or 'audio/mpeg')
    ext='.wav' if 'wav' in mime else '.mp3'
    outfile='pmco_generated'+ext
    with open(outfile,'wb') as f: f.write(data)
    display(Audio(filename=outfile))
    print('저장 파일:',outfile,'MIME:',mime)
else:
    print('음원 생성 건너뜀: 프롬프트와 매핑 결과까지만 구현했습니다.')

## ⑨ KNIME용 결과 CSV 생성
CSV는 음악 품질의 검증 결과가 아니라 매핑 출력 기록이며, 나중에 사람의 평가 점수를 추가합니다.

In [ ]:
row=plan_record(filename,features,music)
row.update({'prompt':prompt,'human_calmness_1to5':'','human_music_fit_1to5':'','notes':''})
result=pd.DataFrame([row]); display(result.T)
result.to_csv('pmco_evaluation.csv',index=False,encoding='utf-8-sig')
print('저장:', 'pmco_evaluation.csv')

## ⑩ 공식 API로 크롤링 실습: Commons 이미지 *메타데이터*
수업의 Requests/BeautifulSoup을 라이선스 메타데이터 수집에 응용합니다. 실제 다운로드와 공개 사용은 각 이미지 라이선스 확인 후 진행하세요.

In [ ]:
import requests
from bs4 import BeautifulSoup
params={'action':'query','format':'json','generator':'categorymembers','gcmtitle':'Category:Landscapes',
        'gcmtype':'file','gcmlimit':10,'prop':'imageinfo','iiprop':'url|extmetadata','iiurlwidth':640}
try:
    response=requests.get('https://commons.wikimedia.org/w/api.php',params=params,
        headers={'User-Agent':'PMCO-Classroom/0.1 (educational metadata research)'},timeout=30)
    response.raise_for_status()
    data=response.json()
    records=[]
    for page in data.get('query',{}).get('pages',{}).values():
        info=(page.get('imageinfo') or [{}])[0]
        meta=info.get('extmetadata') or {}
        artist=BeautifulSoup((meta.get('Artist') or {}).get('value',''),'html.parser').get_text(' ',strip=True)
        records.append({'file':page.get('title'),'url':info.get('thumburl'),
                        'page':info.get('descriptionurl'),'artist':artist,
                        'license':(meta.get('LicenseShortName') or {}).get('value','')})
    commons_df=pd.DataFrame(records)
    display(commons_df.head())
    commons_df.to_csv('commons_metadata.csv',index=False,encoding='utf-8-sig')
except Exception as exc:
    print('Commons API 네트워크/권한 오류:',type(exc).__name__,str(exc)[:250])
    print('크롤링은 선택 실습이므로 이미지 매핑과 API 부분은 계속 사용할 수 있습니다.')

## ⑪ GitHub 저장 (루브릭: 주기적 커밋·브랜치)
1. 노트북 이름을 `PMCO_Colab_01.ipynb`로 바꿉니다.
2. **API 키·키 값이 노출된 셀 출력이 없는지 확인**하고, 파일 메뉴에서 `GitHub에 사본 저장`을 선택합니다.
3. 기존 저장소 `wkwkwwk0712-debug/wonkyu_generative-AI` 또는 새 PMCO 저장소를 선택하세요.
4. 커밋 메시지 `Add interpretable image-to-music mapping notebook`으로 저장합니다.
5. GitHub에서 `README.md`, 이후 `feature/crawling` 브랜치와 PR을 추가해 개발 이력을 남깁니다.
6. Colab에서 다운로드한 CSV를 KNIME으로 열어 기술 및 평가 결과를 분석합니다.

수업에서 이미 배운 `Git add → commit → push`, `branch → merge`와 연결할 수 있습니다.